# Ablation 02 — k (Sparsity) Sweep with Null-Calibrated Jaccard

**Goal.** The baseline SAE (dict_size=4096, k=32) reports a cross-seed mean index-Jaccard of **0.0038**. This notebook tests whether that number carries *any* signal by comparing it against the **exact analytic null** — the expected index-Jaccard between two independent size-`k` subsets of a `dict_size`-set (hypergeometric). The defensible claim is: **the baseline sits exactly on the random-overlap floor** (signal-to-null ratio ~1).

**Design.** Sweep `k ∈ {8, 16, 32, 64}` at **fixed `dict_size=2048`** so the only confound that changes is sparsity (LR is auto-scaled from `dict_size` only, so it stays constant across all groups — eliminating the a1 confound). For each `k`-group we train **4 seeds** (0, 42, 123, 456) for 12000 steps and compute:

1. **Within-group reconstruction** — cosine, variance-explained, MSE, L0 (must equal `k`), dead%.
2. **Within-group index-Jaccard** — `SAEManager.compute_stability` over the 4 seeds of *that group only*, with `n=k` passed explicitly.
3. **Exact hypergeometric null** `null(k, D)` — `Σ_j j/(2k−j)·P(intersection=j)` over the hypergeometric intersection distribution.
4. **Signal-to-null ratio** = raw Jaccard / exact null, with bootstrap 95% CI over the 1494 test samples; report whether the CI excludes 1.
5. **Consensus reappearance** — co-primary, pooled across the 4 seeds of the group.

**Baseline anchor.** The baseline (dict_size=4096, k=32) is reported as a **standalone null-calibrated point** (`null(32,4096)=0.00392`, raw 0.0038 → ratio ~0.97). It is **never** compared via Jaccard against these groups (different `dict_size` — cross-config Jaccard is forbidden by protocol). It is plotted as its own labeled anchor.

**Pre-registered hypothesis.** Signal-to-null ratio ~1 near the baseline sparsity (k=32), **rising as `k` shrinks** (smaller active sets leave fewer chances to overlap by chance → higher ratio if concepts are real), and **dead% rises at very small `k`** (fewer features fire each pass → more features never activate). The Pareto front (variance-explained vs signal-to-null) picks the sweet spot.

**Methodological protocol (hard).**
- *Within-group Jaccard only:* never compute Jaccard across different `dict_size`/`k`. Each `k`-group gets its own `compute_stability` call with its own models + `config={'dict_size':2048,'k':K}` + `n=K`.
- *Output-dir isolation:* all models/results/figures go to per-group `a4` subdirs — never the baseline `models/` or `results/`.
- *Test-set discipline:* stability/Jaccard/naming use test embeddings only.
- *Safe deserialization:* variant model weights loaded via `SAEManager.load` (which uses `utils.load_state_dict`, `weights_only=True`).


## 0. Setup & Configuration

In [ ]:
import os
# Reproducibility — set BEFORE importing torch.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("PYTHONHASHSEED", "0")  # best-effort inside a kernel

import sys
import json
import math
from pathlib import Path

import numpy as np
import torch

# Resolve project root (walk up until 'src/' exists)
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

print(f'Project root: {PROJECT_ROOT}')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {DEVICE}')


In [ ]:
import config

# ── OUTPUT-DIR ISOLATION (hard protocol #2) ────────────────────────────────
# PathsConfig is a MUTABLE @dataclass: override in place to per-group a4 dirs.
ABLATION_ROOT = "ablation_a4"
MODELS_A4     = config.paths.models_dir / ABLATION_ROOT
RESULTS_A4    = config.paths.results_dir / "ablation"

# Unified figures dir across all ablation notebooks (EXACT path).
FIGURES_DIR   = PROJECT_ROOT / "results" / "figures" / "ablation"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_A4    = FIGURES_DIR
for d in (MODELS_A4, RESULTS_A4, FIGURES_A4):
    d.mkdir(parents=True, exist_ok=True)

# Input embeddings come from the committed splits (NOT rebuilt — protocol #3).
TRAIN_EMB = config.paths.train_embeddings_path
TEST_EMB  = config.paths.test_embeddings_path

# ── Ablation grid (dict_size FIXED → LR auto-scale is constant across groups) ─
DICT_SIZE     = 2048
K_GRID        = (8, 16, 32, 64)
ABLATION_SEEDS = (0, 42, 123, 456)
N_STEPS       = 12_000
BATCH_SIZE    = config.sae.batch_size
ACTIVATION_DIM = config.sae.activation_dim

print('=== Ablation 02 — k sweep (dict_size FIXED) ===')
print(f'dict_size      = {DICT_SIZE}  (FIXED across all k-groups)')
print(f'k grid         = {K_GRID}')
print(f'seeds          = {ABLATION_SEEDS}  (primary for naming: 42)')
print(f'steps / batch  = {N_STEPS} / {BATCH_SIZE}')
print(f'lr             = auto (~{2e-4 / math.sqrt(DICT_SIZE/16384):.1e}; constant across groups)')
print(f'models dir     = {MODELS_A4}')
print(f'results dir    = {RESULTS_A4}')
print(f'figures dir    = {FIGURES_A4}')


In [ ]:
import utils

# Verify inputs (committed splits + vocab) exist.
for name, p in [('train_embeddings', TRAIN_EMB), ('test_embeddings', TEST_EMB)]:
    ok = Path(p).exists()
    t = utils.load_tensor(p) if ok else None
    print(f'  [{"OK" if ok else "MISSING"}] {name}: {t.shape if ok else p}')

test_emb = utils.load_tensor(TEST_EMB)
train_emb = utils.load_tensor(TRAIN_EMB)
N_TEST = test_emb.shape[0]
print(f'\nN_test = {N_TEST}  (used for bootstrap CI)')


## 1. SAE Training — (k × seed) grid

Train one `TopK` SAE per `(k, seed)` pair. `dict_size` is held at 2048 so the auto-scaled LR is **identical** across all k-groups (eliminating the a1 dict_size→LR confound). Each k-group writes to its own subdir `models/ablation_a4/dict2048_k{K}/sae_seed{seed}/` so seeds sharing the same integer never collide on the `sae_seed{N}/` leaf.

`SAEManager` is constructed per model with the k-group's `dict_size`/`k`; `train()` hardcodes `TopKTrainer` and uses `auxk_alpha` / `dead_feature_threshold` library defaults (1/32 and 10,000,000 respectively).

In [ ]:
from autoencoder.sae_module import SAEManager

trained = {}  # trained[k] = {seed: model_dir}
for K in K_GRID:
    trained[K] = {}
    group_dir = MODELS_A4 / f'dict2048_k{K}'
    group_dir.mkdir(parents=True, exist_ok=True)
    for seed in ABLATION_SEEDS:
        ae_path = group_dir / f'sae_seed{seed}' / 'trainer_0' / 'ae.pt'
        alt     = group_dir / f'sae_seed{seed}' / 'ae.pt'
        if ae_path.exists() or alt.exists():
            print(f'[skip] k={K} seed={seed} already trained')
            trained[K][seed] = group_dir / f'sae_seed{seed}'
            continue
        print(f'\n--- Training k={K} seed={seed} (dict_size={DICT_SIZE}, {N_STEPS} steps) ---')
        mgr = SAEManager({
            'device': DEVICE,
            'activation_dim': ACTIVATION_DIM,
            'dict_size': DICT_SIZE,
            'k': K,
            'warmup_steps': config.sae.warmup_steps,
        })
        model_dir = mgr.train(
            embeddings_path=TRAIN_EMB,
            seed=seed,
            save_dir=str(group_dir),
            steps=N_STEPS,
            batch_size=BATCH_SIZE,
        )
        trained[K][seed] = model_dir
        print(f'Saved: {model_dir}')

print('\nAll groups trained. Summary:')
for K in K_GRID:
    print(f'  k={K}: {len(trained[K])} seeds')


## 2. Per-group reconstruction metrics

For each k-group (averaged over its 4 seeds, evaluated on the **test** embeddings):
- **cosine** reconstruction similarity (mean per-sample, `compute_cosine_reconstruction`),
- **MSE** (`compute_reconstruction_mse`),
- **variance-explained** = `1 - mse / var(x)` (variance of the *test* embeddings),
- **L0** (must equal `k` — `AutoEncoderTopK` enforces top-k),
- **dead%** (activation-based: a feature never nonzero across all test samples).

These are the reconstruction-quality inputs to the Pareto front.

In [ ]:
def variance_explained(mgr, x):
    """1 - MSE / Var(x), variance computed over the flattened test tensor."""
    with torch.no_grad():
        x_hat = mgr._ae(x.to(mgr._device))
    mse = ((x_hat - x.to(mgr._device)) ** 2).mean().item()
    var = x.to(mgr._device).var().item()
    return 1.0 - mse / var, mse

per_k_metrics = {}
test_var = test_emb.var().item()
print(f'Test-embedding variance (denominator for VE): {test_var:.6f}\n')

for K in K_GRID:
    cosines, mses, ves, l0s, deads = [], [], [], [], []
    for seed in ABLATION_SEEDS:
        mgr = SAEManager({'device': DEVICE, 'dict_size': DICT_SIZE, 'k': K})
        mgr.load(trained[K][seed])
        cos = mgr.compute_cosine_reconstruction(test_emb)
        mse = mgr.compute_reconstruction_mse(test_emb)
        ve = 1.0 - mse / test_var
        sp = mgr.compute_sparsity_metrics(test_emb)
        cosines.append(cos); mses.append(mse); ves.append(ve)
        l0s.append(sp['l0_mean']); deads.append(sp['dead_features_pct'])
        del mgr._ae; mgr._ae = None
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
    per_k_metrics[K] = {
        'cosine': float(np.mean(cosines)),
        'mse': float(np.mean(mses)),
        'var_explained': float(np.mean(ves)),
        'l0_mean': float(np.mean(l0s)),
        'dead_pct': float(np.mean(deads)),
    }
    print(f'k={K:>2}: cosine={np.mean(cosines):.4f}  VE={np.mean(ves):.4f}  '
          f'MSE={np.mean(mses):.2e}  L0={np.mean(l0s):.1f} (expect {K})  dead%={np.mean(deads):.1f}')


## 3. Within-group index-Jaccard + exact hypergeometric null

**Within-group Jaccard (hard protocol #1).** For each k-group we call `SAEManager.compute_stability` over **only that group's 4 seeds**, passing `config={'dict_size':2048,'k':K}` and **`n=K` explicitly**. (`compute_stability` truncates the top-k index tensor to `indices[:, :n]`; the default `n=config['k']` is correct here, but we pass it explicitly because the k=8/16/64 groups must NOT silently under/over-fill.)

**Exact null.** For two independent size-`k` subsets of a `D`-set, the intersection size `j` follows `Hypergeom(M=D, n=k, N=k)`. `compute_stability` averages per-sample Jaccard over samples then over seed pairs (a mean-of-ratios), so the exact expected per-sample Jaccard under the null is:

`null(k, D) = Σ_{j=0}^{k} [ j / (2k − j) ] · P(j)`,   `P(j) = C(k,j)·C(D−k, k−j) / C(D,k)`

implemented exactly via `scipy.stats.hypergeom`.

In [ ]:
from scipy.stats import hypergeom
from math import comb

def exact_null_jaccard(k, D):
    """E[per-sample index-Jaccard] for two independent size-k subsets of a D-set.

    Mean-of-ratios convention (matches compute_stability: per-sample Jaccard
    averaged over samples, then over seed pairs). Intersection size j ~ Hypergeom(D, k, k).
    """
    total = 0.0
    for j in range(0, k + 1):
        p_j = hypergeom.pmf(j, D, k, k)            # P(intersection = j)
        if p_j <= 0:
            continue
        jac_j = j / (2 * k - j) if (2 * k - j) > 0 else 0.0
        total += jac_j * p_j
    return total

# Sanity: closed-form check for tiny case (k=1, D=2): two singletons of a 2-set.
# P(j=0)=0.5 -> 0/2=0 ; P(j=1)=0.5 -> 1/1=1 ; null = 0.5.
assert abs(exact_null_jaccard(1, 2) - 0.5) < 1e-9
# k=2, D=4: enumerate to confirm.
_enum = 0.0
for a in range(4):
    for b in range(4):
        A = {a}; pass
print('hypergeom sanity: null(1,2) =', exact_null_jaccard(1, 2), '(expect 0.5)')

nulls = {K: exact_null_jaccard(K, DICT_SIZE) for K in K_GRID}
# Baseline anchor null (dict_size=4096, k=32) — computed, NOT via within-group Jaccard.
BASE_NULL = exact_null_jaccard(32, 4096)
BASE_RAW  = 0.0038  # committed baseline cross-seed mean index-Jaccard

print('\nExact hypergeometric nulls (per-sample-averaged convention):')
for K in K_GRID:
    print(f'  null(k={K:>2}, D={DICT_SIZE}) = {nulls[K]:.6f}')
print(f'  BASELINE null(k=32, D=4096) = {BASE_NULL:.6f}  (raw 0.0038 -> ratio {BASE_RAW/BASE_NULL:.3f})')


In [ ]:
# Within-group Jaccard per k-group (own 4 seeds ONLY).
within_group = {}
for K in K_GRID:
    model_dirs = [str(trained[K][s]) for s in ABLATION_SEEDS]
    print(f'\nk={K}: compute_stability over {len(model_dirs)} seeds, n={K}')
    stab = SAEManager.compute_stability(
        model_dirs, test_emb,
        config={'device': DEVICE, 'dict_size': DICT_SIZE, 'k': K},
        n=K,
    )
    within_group[K] = stab
    print(f'  mean Jaccard = {stab["mean_jaccard"]:.6f}   '
          f'(null = {nulls[K]:.6f}, ratio = {stab["mean_jaccard"]/nulls[K]:.3f})')


### 3.1 Per-sample Jaccard bootstrap → signal-to-null ratio 95% CI

The aggregate `mean_jaccard` hides the per-sample distribution. To get a CI on the **signal-to-null ratio**, we recompute per-sample Jaccard for every seed pair from the raw top-`k` index sets, resample the 1494 test indices 1000×, and take the 2.5/97.5 percentiles of `mean_ratio = (1/n_pairs)·Σ_pairs mean_s [J_i(s)∪J_j(s)] / null(k, D)`.

In [ ]:
def per_sample_jaccard_for_group(K, n_samples=N_TEST):
    """Return (n_seeds, n_seeds, n_samples) upper-triangular-mean-per-sample array
    of per-sample Jaccard (mean over the C(n_seeds,2) seed pairs, per sample)."""
    n_seeds = len(ABLATION_SEEDS)
    # Top-k index sets per seed per sample.
    sets = []
    for seed in ABLATION_SEEDS:
        mgr = SAEManager({'device': DEVICE, 'dict_size': DICT_SIZE, 'k': K})
        mgr.load(trained[K][seed])
        per_s = []
        # chunk to avoid OOM
        for i in range(0, n_samples, 512):
            chunk = test_emb[i:i+512].to(DEVICE)
            _, _, idx = mgr.encode_topk(chunk)   # (B, K) unsorted indices
            per_s.extend([set(row.tolist()) for row in idx.cpu()])
        sets.append(per_s)
        del mgr._ae; mgr._ae = None
        try: torch.cuda.empty_cache()
        except Exception: pass
    # per-sample mean Jaccard averaged over all seed pairs.
    pair_means = np.zeros(n_samples)
    n_pairs = 0
    for a in range(n_seeds):
        for b in range(a + 1, n_seeds):
            for s in range(n_samples):
                u = sets[a][s] | sets[b][s]
                inter = len(sets[a][s] & sets[b][s])
                pair_means[s] += inter / len(u) if u else 0.0
            n_pairs += 1
    pair_means /= n_pairs
    return pair_means

rng = np.random.default_rng(0)
boot = {}
for K in K_GRID:
    ps = per_sample_jaccard_for_group(K)            # (N_TEST,) per-sample mean Jaccard
    ratios = ps / nulls[K]                           # per-sample signal-to-null ratio
    # bootstrap over test samples
    samp = np.empty(1000)
    idx_all = np.arange(N_TEST)
    for b in range(1000):
        bs = rng.choice(idx_all, size=N_TEST, replace=True)
        samp[b] = ratios[bs].mean()
    lo, hi = np.percentile(samp, [2.5, 97.5])
    raw = within_group[K]['mean_jaccard']
    boot[K] = {
        'raw_jaccard': float(raw),
        'exact_null': float(nulls[K]),
        'signal_to_null': float(raw / nulls[K]),
        'ci_low': float(lo),
        'ci_high': float(hi),
        'excludes_one': bool(lo > 1.0),
    }
    print(f'k={K:>2}: raw={raw:.5f}  null={nulls[K]:.5f}  ratio={raw/nulls[K]:.3f}  '
          f'95% CI [{lo:.3f}, {hi:.3f}]  excludes 1? {lo > 1.0}')


### 3.2 Co-primary: consensus reappearance (connected_components, mirrors a0)

Index-agnostic cross-seed robustness, computed with the **same algorithm as
ablation 00**: within each k-group, pool the LIVE decoder rows across the 4 seeds
(drop dead rows with norm `< 1e-8`), L2-normalize, build a sparse `cosine > 0.90`
adjacency (strict `>`), and run `scipy.sparse.csgraph.connected_components`.
`consensus_rate(K, min_seeds)` is the fraction of pooled LIVE rows that belong to
a cluster spanning `>= min_seeds` of the 4 seeds. Using the identical algorithm
across a0 / a1 / a4 makes the cross-ablation consensus rates directly comparable
(a0's empirical result was consensus@high-tau ~ 0 — a null; the same is expected
here, and is reported as such, not as a positive result).

In [ ]:
def consensus_rate(K, min_seeds=2):
    """Connected-components consensus on pooled LIVE decoder rows (matches
    ablation 00's algorithm EXACTLY), within this k-group's 4 seeds.

    Pool the LIVE decoder rows (drop dead rows with norm < 1e-8), L2-normalize,
    build a sparse `cosine > tau` adjacency (strict > tau), run
    `scipy.sparse.csgraph.connected_components`, then count clusters spanning
    >= min_seeds of the 4 seeds. Returns the fraction of LIVE rows that belong
    to a cluster spanning >= min_seeds (a0's row-based consensus_rate(m))."""
    DEAD_THRESHOLD = 1e-8
    TAU = 0.90  # cosine boundary, STRICT (> tau); matches ablation 00
    import torch.nn.functional as F
    from scipy import sparse
    from scipy.sparse.csgraph import connected_components

    rows_list, seed_labels = [], []
    for seed in ABLATION_SEEDS:
        mgr = SAEManager({'device': DEVICE, 'activation_dim': ACTIVATION_DIM,
                          'dict_size': DICT_SIZE, 'k': K})
        mgr.load(trained[K][seed])
        W = mgr.get_decoder_weights().cpu()        # (dict_size, 512)
        norms = W.norm(dim=1)
        live = (norms >= DEAD_THRESHOLD).nonzero(as_tuple=True)[0]
        rows_list.append(W[live].clone())
        seed_labels.append(np.full(len(live), seed, dtype=np.int64))
        del mgr._ae; mgr._ae = None
        try: torch.cuda.empty_cache()
        except Exception: pass
    rows_n = F.normalize(torch.cat(rows_list, dim=0).float(), dim=1) \
                 .numpy().astype(np.float64)
    labels = np.concatenate(seed_labels)

    cos_full = rows_n @ rows_n.T
    np.fill_diagonal(cos_full, -1.0)
    adj = sparse.csr_matrix(cos_full > TAU)              # STRICT > tau
    n_components, cluster_labels = connected_components(csgraph=adj, directed=False)

    # per-row count of distinct seeds in its cluster (a0's consensus_rate(m))
    row_seedcount = np.empty(len(labels), dtype=np.int64)
    for c in range(n_components):
        member_seeds = np.unique(labels[cluster_labels == c])
        row_seedcount[cluster_labels == c] = len(member_seeds)
    return float((row_seedcount >= min_seeds).mean())

consensus = {}
for K in K_GRID:
    consensus[K] = {
        'consensus_rate_ge2': consensus_rate(K, min_seeds=2),
        'consensus_rate_ge3': consensus_rate(K, min_seeds=3),
    }
    print(f'k={K:>2}: consensus reappearance (connected_components, cos>0.90)  '
          f'>=2 seeds: {consensus[K]["consensus_rate_ge2"]:.3f}  '
          f'>=3 seeds: {consensus[K]["consensus_rate_ge3"]:.3f}')

## 4. Figures

**(1) k-vs-stability:** x = k, y = raw within-group Jaccard with the exact-null line overlaid; secondary axis = signal-to-null ratio with bootstrap 95% CI bars; baseline (dict4096/k32) marked as its own labeled anchor.

**(2) Pareto front:** x = variance-explained, y = signal-to-null ratio, one point per k, baseline anchor marked — best-k maximizes stability ratio without collapsing reconstruction.

In [ ]:
import matplotlib
matplotlib.use('Agg')  # safe in headless; remove for inline
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

ks = list(K_GRID)
raws  = [boot[K]['raw_jaccard'] for K in ks]
nulls_line = [nulls[K] for K in ks]
ratios = [boot[K]['signal_to_null'] for K in ks]
ci_lo  = [boot[K]['ci_low'] for K in ks]
ci_hi  = [boot[K]['ci_high'] for K in ks]
err_lo = [r - lo for r, lo in zip(ratios, ci_lo)]
err_hi = [hi - r for hi, r in zip(ci_hi, ratios)]

# ── Figure 1: k vs stability (dual axis) ────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(9, 5.5))
ax1.plot(ks, raws, 'o-', color='steelblue', label='Raw within-group Jaccard', markersize=8)
ax1.plot(ks, nulls_line, 's--', color='gray', label='Exact null (hypergeometric)', markersize=7)
# baseline anchor
ax1.scatter([32], [BASE_RAW], marker='*', s=260, color='crimson', zorder=5,
            label=f'Baseline (k=32, D=4096): raw={BASE_RAW:.4f}')
ax1.set_xlabel('k (sparsity)')
ax1.set_ylabel('Mean index-Jaccard', color='steelblue')
ax1.set_xticks(ks)
ax1.set_title('k sweep — within-group stability vs exact null (dict_size=2048)')
ax1.tick_params(axis='y', labelcolor='steelblue')

ax2 = ax1.twinx()
ax2.errorbar(ks, ratios, yerr=[err_lo, err_hi], fmt='^-', color='darkgreen',
             label='Signal-to-null ratio (95% CI)', markersize=8, capsize=4)
ax2.axhline(1.0, color='black', linestyle=':', alpha=0.6, label='Ratio = 1 (null)')
ax2.axhline(BASE_RAW / BASE_NULL, color='crimson', linestyle=':', alpha=0.5,
            label=f'Baseline ratio = {BASE_RAW/BASE_NULL:.2f}')
ax2.set_ylabel('Signal-to-null ratio', color='darkgreen')
ax2.tick_params(axis='y', labelcolor='darkgreen')

lines1, lab1 = ax1.get_legend_handles_labels()
lines2, lab2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, lab1 + lab2, loc='upper left', fontsize=8)
plt.tight_layout()
f1 = FIGURES_A4 / 'a4_k_vs_stability.png'
plt.savefig(f1, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {f1}')


In [ ]:
# ── Figure 2: Pareto front (variance-explained vs signal-to-null) ──────────
ves = [per_k_metrics[K]['var_explained'] for K in ks]

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.errorbar(ves, ratios, xerr=None, yerr=[err_lo, err_hi], fmt='o-', color='darkgreen',
            markersize=10, capsize=4, label='k-groups (dict_size=2048)')
for K, x, y in zip(ks, ves, ratios):
    ax.annotate(f'k={K}', (x, y), textcoords='offset points', xytext=(8, 8), fontsize=10)
# baseline anchor
ax.scatter([0.993], [BASE_RAW / BASE_NULL], marker='*', s=260, color='crimson', zorder=5,
           label=f'Baseline (D=4096, k=32): VE=0.993, ratio={BASE_RAW/BASE_NULL:.2f}')
ax.axhline(1.0, color='black', linestyle=':', alpha=0.5)
ax.set_xlabel('Variance explained (1 − MSE/Var)')
ax.set_ylabel('Signal-to-null ratio')
ax.set_title('Pareto front — reconstruction vs null-calibrated stability')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
f2 = FIGURES_A4 / 'a4_pareto_front.png'
plt.savefig(f2, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {f2}')


## 5. Save results

Write `results/ablation/a4_k_sweep.json` with per-k metrics, the null calibration, the baseline anchor, and the pre-registered hypothesis verdict.

In [ ]:
results = {
    'ablation': '02_k_sweep',
    'protocol': {
        'dict_size_fixed': DICT_SIZE,
        'k_grid': list(K_GRID),
        'seeds': list(ABLATION_SEEDS),
        'steps': N_STEPS,
        'lr': 'auto (constant across groups; scales with dict_size only)',
        'n_test': N_TEST,
        'jaccard': 'within-group only; compute_stability n=k passed explicitly',
        'null': 'exact hypergeometric, mean-of-ratios convention',
        'ci': 'bootstrap 1000x over test samples',
    },
    'per_k': {},
    'baseline_anchor': {
        'dict_size': 4096, 'k': 32,
        'raw_jaccard': BASE_RAW,
        'exact_null': float(BASE_NULL),
        'signal_to_null': float(BASE_RAW / BASE_NULL),
        'note': 'standalone null-calibrated point; NOT compared via Jaccard (different dict_size)',
    },
}

for K in K_GRID:
    results['per_k'][str(K)] = {
        **boot[K],
        **consensus[K],
        **per_k_metrics[K],
    }

out_path = RESULTS_A4 / 'a4_k_sweep.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved: {out_path}')

# Hypothesis verdict table
print('\n=== Signal-to-null ratio summary ===')
print(f"{'k':>4} {'raw':>10} {'null':>10} {'ratio':>8} {'95% CI':>20} {'excl1':>6} {'VE':>7} {'dead%':>7}")
for K in K_GRID:
    b = boot[K]
    print(f"{K:>4} {b['raw_jaccard']:>10.5f} {b['exact_null']:>10.5f} "
          f"{b['signal_to_null']:>8.3f} [{b['ci_low']:.3f}, {b['ci_high']:.3f}]"
          f"{'':<2}{str(b['excludes_one']):>6} "
          f"{per_k_metrics[K]['var_explained']:>7.4f} {per_k_metrics[K]['dead_pct']:>7.1f}")
print(f"\nBaseline anchor: ratio = {BASE_RAW/BASE_NULL:.3f} (raw {BASE_RAW}, null {BASE_NULL:.5f})")


## 6. Interpretation

- **If the baseline anchor ratio is ~1** (raw 0.0038 / null 0.00392 ≈ 0.97), the baseline cross-seed Jaccard is **indistinguishable from random index overlap** — the defensible 6A claim.
- **Rising ratio at smaller k** means: with fewer active features, chance overlap shrinks faster than real overlap, so the *relative* signal grows — concepts are more reproducible per-active-slot than raw Jaccard suggests.
- **Pareto front:** the best-k maximizes signal-to-null without collapsing variance-explained. Expect a smaller-k sweet spot if reconstruction holds up; if VE drops sharply at k=8/16, k=32 remains the operating point.
- **Dead% rises at very small k** because fewer features fire each pass → more features never activate on the test set.
- **Consensus reappearance** is the index-agnostic corroboration: a high ≥2-seed rate paired with a near-1 ratio means features *are* reproducible even though raw index-Jaccard looks like noise.
